- SIH 26190 — FIR Key Information Extraction (LayoutLMv3 fine-tune)
- Trains a layout-aware model to tag each word/text-span in an FIR
- image as: Police Station / Year / Statutes / Complainant's Name / Other

# 1. Install dependencies

In [ ]:
!pip install -q transformers datasets seqeval Pillow torch torchvision accelerate

import os, json, random
import torch
from PIL import Image
from transformers import (
    LayoutLMv3Processor, LayoutLMv3ForTokenClassification,
    TrainingArguments, Trainer
)
from datasets import Dataset as HFDataset
import numpy as np
from seqeval.metrics import classification_report, f1_score

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done



# 2. Clone the dataset repo

In [ ]:
!git clone -q https://github.com/LegalDocumentProcessing/FIR_Dataset_ICDAR2023.git
DATA_DIR = "FIR_Dataset_ICDAR2023"
IMG_DIR  = os.path.join(DATA_DIR, "FIR_images_v1")

with open(os.path.join(DATA_DIR, "FIR_details.json"), "r") as f:
    raw_annotations = json.load(f)

# raw_annotations is a list of {image_id, bbox, category_id, image_name, text}
# Group annotations by image so each image becomes one training example
CATEGORY_NAMES = {0: "POLICE_STATION", 1: "YEAR", 2: "STATUTES", 3: "COMPLAINANT_NAME"}
LABEL_LIST = ["O"] + [f"B-{v}" for v in CATEGORY_NAMES.values()]  # BIO scheme
label2id = {l: i for i, l in enumerate(LABEL_LIST)}
id2label = {i: l for l, i in label2id.items()}

images_map = {}
for ann in raw_annotations:
    img_name = ann["image_name"]
    images_map.setdefault(img_name, []).append(ann)

image_names = list(images_map.keys())
random.seed(42)
random.shuffle(image_names)
split_idx = int(0.85 * len(image_names))
train_names, val_names = image_names[:split_idx], image_names[split_idx:]

def normalize_bbox(bbox, width, height):
    # LayoutLMv3 expects bboxes normalized to a 0-1000 scale
    xmin, ymin, xmax, ymax = bbox
    return [
        int(1000 * xmin / width), int(1000 * ymin / height),
        int(1000 * xmax / width), int(1000 * ymax / height),
    ]

def build_example(img_name):
    img_path = os.path.join(IMG_DIR, img_name)
    image = Image.open(img_path).convert("RGB")
    width, height = image.size

    words, boxes, ner_tags = [], [], []
    for ann in images_map[img_name]:
        text = ann["text"].strip()
        if not text:
            continue
        cat = CATEGORY_NAMES.get(ann["category_id"])
        box = normalize_bbox(ann["bbox"], width, height)
        # split multi-word annotated text into individual words, same box reused
        # (LayoutLMv3 needs word-level granularity; this is an approximation
        # since we only have span-level boxes, not per-word boxes)
        for i, w in enumerate(text.split()):
            words.append(w)
            boxes.append(box)
            ner_tags.append(label2id[f"B-{cat}"] if cat else label2id["O"])

    return {"image": image, "words": words, "boxes": boxes, "ner_tags": ner_tags}

train_examples = [build_example(n) for n in train_names]
val_examples   = [build_example(n) for n in val_names]

# 3. Processor + encoding

In [ ]:
processor = LayoutLMv3Processor.from_pretrained(
    "microsoft/layoutlmv3-base", apply_ocr=False  # we already have words+boxes
)

def encode(example):
    encoding = processor(
        example["image"],
        example["words"],
        boxes=example["boxes"],
        word_labels=example["ner_tags"],
        truncation=True,
        padding="max_length",
        max_length=512,
        return_tensors="pt",
    )
    return {k: v.squeeze(0) for k, v in encoding.items()}

train_dataset = [encode(ex) for ex in train_examples]
val_dataset   = [encode(ex) for ex in val_examples]

class FIRDataset(torch.utils.data.Dataset):
    def __init__(self, data): self.data = data
    def __len__(self): return len(self.data)
    def __getitem__(self, idx): return self.data[idx]

train_ds = FIRDataset(train_dataset)
val_ds   = FIRDataset(val_dataset)

preprocessor_config.json:   0%|          | 0.00/275 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/856 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.14k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

# 4. Model

In [ ]:
model = LayoutLMv3ForTokenClassification.from_pretrained(
    "microsoft/layoutlmv3-base",
    num_labels=len(LABEL_LIST),
    id2label=id2label,
    label2id=label2id,
)

model.safetensors: reconstructing file:   0%|          |  0.00B /  501MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/212 [00:00<?, ?it/s]

[transformers] LayoutLMv3ForTokenClassification LOAD REPORT from: microsoft/layoutlmv3-base
Key               | Status  | 
------------------+---------+-
classifier.bias   | MISSING | 
classifier.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


# 5. Metrics

In [ ]:
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=2)
    true_preds, true_labels = [], []
    for pred_row, label_row in zip(predictions, labels):
        p_seq, l_seq = [], []
        for p, l in zip(pred_row, label_row):
            if l == -100:
                continue
            p_seq.append(id2label[p])
            l_seq.append(id2label[l])
        true_preds.append(p_seq)
        true_labels.append(l_seq)
    return {"f1": f1_score(true_labels, true_preds)}


# 6. Training

In [8]:
training_args = TrainingArguments(
    output_dir="./fir_layoutlmv3",
    max_steps=1000,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    learning_rate=1e-5,
    eval_strategy="steps",
    eval_steps=100,
    save_steps=200,
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
)

trainer.train()

Step,Training Loss,Validation Loss,F1
100,0.000096,0.027452,0.997478
200,0.032011,0.000063,1.000000
300,0.000059,0.000030,1.000000
400,0.000050,0.000065,1.000000
500,0.000045,0.000035,1.000000
600,0.000044,0.001864,0.998739
700,0.000037,0.002025,0.998739
800,0.000035,0.001734,0.998739
900,0.000034,0.001787,0.998739
1000,0.000033,0.001712,0.998739


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1000, training_loss=0.0016544452308444306, metrics={'train_runtime': 907.6872, 'train_samples_per_second': 2.203, 'train_steps_per_second': 1.102, 'total_flos': 527201236992000.0, 'train_loss': 0.0016544452308444306, 'epoch': 4.329004329004329})


# 7. Save the fine-tuned model

In [9]:

trainer.save_model("./fir_layoutlmv3_final")
processor.save_pretrained("./fir_layoutlmv3_final")
print("Training complete. Model saved to ./fir_layoutlmv3_final")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training complete. Model saved to ./fir_layoutlmv3_final
